# 02 — cGAN Experiments on Fashion-MNIST

Group **A Day in Ali** · CAK4KBB3 Project 1

This notebook runs and inspects the conditional GAN. All reusable logic lives in
`src/`; the notebook only calls it and shows the results:

1. Setup and data check
2. Model size
3. Training (resumes from the latest checkpoint if one exists)
4. Loss curves
5. Fixed-grid progress across epochs
6. Fresh samples per class from the trained generator
7. Observations

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

# Run from the repo root so `src` imports and relative paths work.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import torch
from PIL import Image
from torchvision.utils import make_grid

from src.config import CHECKPOINTS_DIR, CLASS_NAMES, LATENT_DIM, LOGS_DIR, NUM_CLASSES, SAMPLES_DIR, SEED
from src.data import get_dataloader
from src.models import Discriminator, Generator, count_parameters
from src.plots import save_loss_curves
from src.train import train
from src.utils import get_device, load_checkpoint, set_seed

%matplotlib inline
print("Repo root:", ROOT)
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

### Experiment settings

Change these before running. `EPOCHS` is the last epoch to reach: with `RESUME = True`
an existing `outputs/checkpoints/ckpt.pt` is continued instead of starting over, so
re-running the notebook after training finishes does not retrain.

One epoch at batch size 128 takes about 80 seconds on a laptop CPU (≈40 minutes for 30).
Set `MAX_BATCHES` to a small number (e.g. `20`) for a quick smoke run.

In [ ]:
EPOCHS = 30
BATCH_SIZE = 128
LR = 2e-4
SAMPLE_EVERY = 5
RESUME = True
MAX_BATCHES = None  # e.g. 20 for a quick smoke run
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

set_seed(SEED)
device = get_device(DEVICE)
print("Device:", device)

### Data check

One batch of real images, rescaled from `[-1, 1]` back to `[0, 1]` for display.

In [ ]:
loader = get_dataloader(batch_size=BATCH_SIZE, seed=SEED)
images, labels = next(iter(loader))
print("Batch:", tuple(images.shape), "| range:", round(images.min().item(), 3), "to", round(images.max().item(), 3))
print("Labels in batch:", sorted({CLASS_NAMES[i] for i in labels.tolist()}))

grid = make_grid(images[:64], nrow=16, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(12, 3.5))
plt.imshow(grid.permute(1, 2, 0))
plt.axis("off")
plt.title("Real Fashion-MNIST samples")
plt.show()

## 2. Model size

In [ ]:
g_params = count_parameters(Generator())
d_params = count_parameters(Discriminator())
print(f"Generator:     {g_params:>9,}")
print(f"Discriminator: {d_params:>9,}")
print(f"Total:         {g_params + d_params:>9,}")

## 3. Training

Each batch runs one discriminator update and one generator update
(`BCEWithLogitsLoss`, real targets 0.9, fake targets 0.0). Checkpoints, sample grids,
`history.json`, and `loss_curves.png` are written under `outputs/`.

In [ ]:
ckpt_path = CHECKPOINTS_DIR / "ckpt.pt"
resume_from = ckpt_path if RESUME and ckpt_path.exists() else None
print("Resuming from:" if resume_from else "Starting fresh.", resume_from or "")

history = train(
    loader,
    epochs=EPOCHS,
    lr=LR,
    device=device,
    sample_every=SAMPLE_EVERY,
    resume=resume_from,
    seed=SEED,
    max_batches=MAX_BATCHES,
)
print("Epochs recorded:", len(history["loss_d"]))

## 4. Loss curves

Healthy cGAN training usually shows both losses settling into a band rather than one
going to zero. Warning signs: discriminator loss near 0 with accuracy near 1
(D wins, G stops learning), or `D(G(z))` stuck near 0.

In [ ]:
epochs_axis = range(1, len(history["loss_d"]) + 1)
fig, (ax_loss, ax_d) = plt.subplots(1, 2, figsize=(12, 4))
ax_loss.plot(epochs_axis, history["loss_d"], marker="o", label="Discriminator")
ax_loss.plot(epochs_axis, history["loss_g"], marker="o", label="Generator")
ax_loss.set(xlabel="Epoch", ylabel="BCE loss", title="Loss")
ax_loss.legend()
ax_d.plot(epochs_axis, history["acc_d"], marker="o", label="D accuracy")
ax_d.plot(epochs_axis, history["d_real"], linestyle="--", label="D(x) real")
ax_d.plot(epochs_axis, history["d_fake"], linestyle="--", label="D(G(z)) fake")
ax_d.set(xlabel="Epoch", ylim=(0, 1), title="Discriminator")
ax_d.legend()
for ax in (ax_loss, ax_d):
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"{'epoch':>5} {'loss_d':>8} {'loss_g':>8} {'acc_d':>6} {'D(x)':>6} {'D(G(z))':>8}")
for i in epochs_axis:
    j = i - 1
    print(f"{i:>5} {history['loss_d'][j]:>8.4f} {history['loss_g'][j]:>8.4f} "
          f"{history['acc_d'][j]:>6.3f} {history['d_real'][j]:>6.3f} {history['d_fake'][j]:>8.3f}")

## 5. Fixed-grid progress

Every saved grid uses the same noise and labels, so changes between epochs come only
from training. Rows are classes in order (T-shirt/top at the top, Ankle boot at the bottom).

In [ ]:
grid_files = sorted(SAMPLES_DIR.glob("epoch_*.png"))
print("Saved grids:", [p.name for p in grid_files])

if grid_files:
    # Show at most 4 grids, evenly spaced from first to last.
    n = len(grid_files)
    show = [grid_files[i] for i in sorted({round(k * (n - 1) / 3) for k in range(4)})]
    fig, axes = plt.subplots(1, len(show), figsize=(4 * len(show), 5), squeeze=False)
    for ax, path in zip(axes[0], show):
        ax.imshow(Image.open(path), cmap="gray")
        ax.set_title(path.stem.replace("_", " "))
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 6. Fresh samples per class

New random noise (not the fixed grid) through the latest generator, in eval mode.
Each row should clearly show its labelled class and still vary within the row;
identical images across a row would suggest mode collapse.

In [ ]:
generator = Generator().to(device)
ckpt = load_checkpoint(ckpt_path, generator, device=device)
generator.eval()
print("Loaded generator from epoch", ckpt["epoch"])

per_class = 10
gen_rng = torch.Generator().manual_seed(SEED + 1)
z = torch.randn(NUM_CLASSES * per_class, LATENT_DIM, generator=gen_rng).to(device)
y = torch.arange(NUM_CLASSES, device=device).repeat_interleave(per_class)
with torch.no_grad():
    fakes = generator(z, y).cpu()

fig, axes = plt.subplots(NUM_CLASSES, per_class, figsize=(per_class * 1.1, NUM_CLASSES * 1.15))
for row in range(NUM_CLASSES):
    for col in range(per_class):
        ax = axes[row, col]
        ax.imshow(fakes[row * per_class + col, 0], cmap="gray", vmin=-1, vmax=1)
        ax.set_xticks([]); ax.set_yticks([])
        if col == 0:
            ax.set_ylabel(CLASS_NAMES[row], rotation=0, ha="right", va="center", fontsize=9)
plt.suptitle(f"Generated samples, epoch {ckpt['epoch']}")
plt.tight_layout()
plt.show()

## 7. Observations

Fill in after a full run; these notes feed the *Optimasi* section of
`logs/LOG_INTERAKSI_LLM.md` and the final report.

- **Training stability:** _e.g. do losses settle? does D overpower G?_
- **Class quality:** _which classes look best/worst (Sandal and Bag are often hardest)?_
- **Diversity / mode collapse:** _do rows vary?_
- **Next experiment (max two variants):** _e.g. learning rate, label smoothing, no BatchNorm on D's first layer._